In [ ]:
import sys
try:
    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.is_available())
except Exception as e:
    print("Falha ao importar torch:", e)
    raise


In [ ]:
from SEConformer_TL import (
    DEVICE,
    build_inbreast_csv,
    make_run_dirs,
    train_inbreast_transfer_all,
)
print("Device:", DEVICE)


In [ ]:
inbreast_csv = r"C:\\Users\\franc\\OneDrive\\Desktop\\Breast Cancer\\BrestCancer Datasets\\INBreast\\INbreast\\INbreast.csv"
dicom_dir = r"C:\\Users\\franc\\OneDrive\\Desktop\\Breast Cancer\\BrestCancer Datasets\\INBreast\\INbreast\\AllDICOMs"

# pesos do SEConformer treinado em histologia
HISTO_WEIGHTS = r"C:\\Users\\franc\\OneDrive\\Desktop\\Breast Cancer\\results\\SEConformer\\seconformer_20260406-235541\\models\\seconformer.pt"

# Cria o CSV com folds (Bi-Rads >=4 = maligno)
df = build_inbreast_csv(
    inbreast_csv,
    dicom_dir,
    out_csv="INbreast_Folds.csv",
    n_splits=5,
    random_state=42,
    birads_threshold=4,
    mode="multiclass",
)
print("Total imagens:", len(df))


In [ ]:
# treino com TODO o dataset (transfer learning, so ultima camada)
run_dirs = make_run_dirs()

model, run_dirs = train_inbreast_transfer_all(
    "INbreast_Folds.csv",
    weights_path=HISTO_WEIGHTS,
    epochs=5,
    run_dirs=run_dirs,
    img_size=224,
    freeze_backbone=True,
    val_split=0.2,
    balance=True
)

print("Run salvo em:", run_dirs["out_dir"])
